# Pipeline Leakage 多目标检测：YOLO11s + LoRA 微调

本 Notebook 针对 `pipeline_leakage1` 数据集完成以下流程：

1. 检查 YOLO-Master、PyTorch、CUDA 环境；
2. 读取数据集真实 `data.yaml`，不假设原类别 ID；
3. 将多边形标注自动转换为目标检测外接矩形框；
4. 删除 `no leak` 检测框，但保留正常图片并生成空标签；
5. 将最终类别固定为：
   - `crack`
   - `water`
   - `leak`
6. 如果原数据集没有独立 `test`，将现有 `valid` 固定拆分成 `valid + test`；
7. 先运行 3 epochs 冒烟训练；
8. 再运行 150 epochs YOLO11s + LoRA 正式训练；
9. 在独立测试集计算 Precision、Recall、mAP50、mAP50-95；
10. 输出带检测框的测试图片。

> 注意：本 Notebook 依赖腾讯 YOLO-Master 的 LoRA 实现，而不是普通 PyPI 版 Ultralytics。  
> 正常图片不输出 `no leak` BBox；前端应在没有检测到任何目标时显示“当前图像中未检测到明显异常”。

## 运行前环境准备

请先在 **Anaconda PowerShell 或终端** 中执行：

```powershell
conda create -n yolo_master python=3.11 -y
conda activate yolo_master

cd D:\dev\projects
git clone https://github.com/Tencent/YOLO-Master.git
cd YOLO-Master

python -m pip install --upgrade pip
pip install -r requirements.txt
pip install -e .

pip install jupyter ipykernel scikit-learn pandas matplotlib pillow pyyaml
python -m ipykernel install --user --name yolo_master --display-name "Python 3 (yolo_master)"
```

然后从 YOLO-Master 环境启动 Jupyter/PyCharm Notebook。

不要在该环境中再次执行：

```powershell
pip install -U ultralytics
```

否则可能覆盖 YOLO-Master 的 LoRA 版本。

## Cell 1：项目配置

In [ ]:
from pathlib import Path

# ============================================================
# 项目根目录
# ============================================================
PROJECT_ROOT = Path(r"D:\dev\projects\yolo_train")

# ============================================================
# 原始数据集根目录
# 注意：必须指向包含 train、valid、data.yaml 的这一层
# ============================================================
RAW_DATASET_ROOT = (
    PROJECT_ROOT
    / "datasets"
    / "pipeline_leakage1"
)

# Notebook自动处理后的数据
WORKSPACE_ROOT = PROJECT_ROOT / "workspace"

PROCESSED_DATASET_ROOT = (
    WORKSPACE_ROOT
    / "processed_dataset"
)

# 模型训练结果
RUNS_ROOT = PROJECT_ROOT / "models" / "runs"

# 指标和数据处理报告
REPORT_ROOT = WORKSPACE_ROOT / "reports"

# 推理结果图片
DEMO_ROOT = WORKSPACE_ROOT / "demo_predictions"

# 原始数据配置
DATA_YAML_PATH = RAW_DATASET_ROOT / "data.yaml"

# 创建输出目录
for path in [
    WORKSPACE_ROOT,
    PROCESSED_DATASET_ROOT,
    RUNS_ROOT,
    REPORT_ROOT,
    DEMO_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

print("项目根目录：", PROJECT_ROOT)
print("数据集目录：", RAW_DATASET_ROOT)
print("数据配置：", DATA_YAML_PATH)
print("处理后数据：", PROCESSED_DATASET_ROOT)
print("训练输出：", RUNS_ROOT)

## Cell 2：检查 YOLO-Master、LoRA 和 CUDA 环境

In [ ]:
import sys
import inspect
import torch
import ultralytics
from ultralytics import YOLO

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Ultralytics 导入路径:", ultralytics.__file__)
print("CUDA 可用:", torch.cuda.is_available())
print("CUDA Runtime:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU 数量:", torch.cuda.device_count())
else:
    raise RuntimeError("CUDA 不可用。请先正确安装支持当前显卡的 PyTorch CUDA 版本。")

ultralytics_path = str(Path(ultralytics.__file__).resolve()).lower()
if "yolo-master" not in ultralytics_path and "yolo_master" not in ultralytics_path:
    print(
        "\n[警告] 当前 ultralytics 路径看起来不是 YOLO-Master 可编辑安装目录。\n"
        "LoRA 参数可能无法识别。请确认已经在 YOLO-Master 根目录执行 pip install -e ."
    )

# 检查 YOLO-Master 默认配置中是否存在 LoRA 参数
lora_config_available = False
try:
    from ultralytics.cfg import DEFAULT_CFG_DICT
    required_lora_keys = {
        "lora_r",
        "lora_alpha",
        "lora_dropout",
        "lora_gradient_checkpointing",
    }
    available_keys = required_lora_keys.intersection(DEFAULT_CFG_DICT.keys())
    print("检测到的 LoRA 配置项:", sorted(available_keys))
    lora_config_available = "lora_r" in DEFAULT_CFG_DICT
except Exception as exc:
    print("无法读取 DEFAULT_CFG_DICT:", exc)

if not lora_config_available:
    raise RuntimeError(
        "当前 ultralytics 未检测到 lora_r 配置。"
        "请安装腾讯 YOLO-Master，并避免被官方 ultralytics 覆盖。"
    )

print("\n环境检查通过，可以继续。")

## Cell 3：读取并检查原始数据集

In [ ]:
from __future__ import annotations

import json
import re
import shutil
import hashlib
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import yaml
import numpy as np
import pandas as pd
from PIL import Image

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"
}

DATA_YAML_PATH = RAW_DATASET_ROOT / "data.yaml"

if not RAW_DATASET_ROOT.exists():
    raise FileNotFoundError(f"原始数据集目录不存在：{RAW_DATASET_ROOT}")

if not DATA_YAML_PATH.exists():
    yaml_candidates = list(RAW_DATASET_ROOT.glob("*.yaml")) + list(RAW_DATASET_ROOT.glob("*.yml"))
    if not yaml_candidates:
        raise FileNotFoundError(
            f"{RAW_DATASET_ROOT} 下没有找到 data.yaml 或其他 YAML 配置文件"
        )
    DATA_YAML_PATH = yaml_candidates[0]

with DATA_YAML_PATH.open("r", encoding="utf-8") as file:
    raw_data_yaml = yaml.safe_load(file)

print("原 data.yaml 路径：", DATA_YAML_PATH)
print(json.dumps(raw_data_yaml, ensure_ascii=False, indent=2))


def parse_class_names(names_config: Any) -> dict[int, str]:
    if isinstance(names_config, list):
        return {index: str(name) for index, name in enumerate(names_config)}
    if isinstance(names_config, dict):
        return {int(index): str(name) for index, name in names_config.items()}
    raise ValueError("data.yaml 中的 names 必须是列表或字典")


RAW_CLASS_NAMES = parse_class_names(raw_data_yaml.get("names"))
print("\n原始类别：")
for class_id, class_name in RAW_CLASS_NAMES.items():
    print(f"{class_id}: {class_name}")

## Cell 4：定义数据集预处理函数

In [ ]:
from sklearn.model_selection import train_test_split


def normalize_class_name(name: str) -> str:
    """统一类别名称，兼容 no leak / no_leak / No-Leak 等写法。"""
    value = name.strip().lower()
    value = re.sub(r"[_\-]+", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value


TARGET_CLASSES = ["crack", "water", "leak"]
TARGET_CLASS_TO_ID = {
    name: index for index, name in enumerate(TARGET_CLASSES)
}

CLASS_ALIASES = {
    "crack": "crack",
    "cracks": "crack",
    "pipe crack": "crack",
    "severe crack": "crack",
    "water": "water",
    "standing water": "water",
    "visible water": "water",
    "leak": "leak",
    "leakage": "leak",
    "water leak": "leak",
    "pipe leak": "leak",
    "pipeline leak": "leak",
    "no leak": "no_leak",
    "noleak": "no_leak",
}


def map_raw_class_name(raw_name: str) -> str | None:
    normalized = normalize_class_name(raw_name)
    compact = normalized.replace(" ", "")

    if normalized in CLASS_ALIASES:
        mapped = CLASS_ALIASES[normalized]
    elif compact == "noleak":
        mapped = "no_leak"
    else:
        # 对未知类别不进行盲目映射
        return None

    return mapped


RAW_ID_TO_TARGET_NAME: dict[int, str | None] = {}

for raw_id, raw_name in RAW_CLASS_NAMES.items():
    mapped_name = map_raw_class_name(raw_name)
    RAW_ID_TO_TARGET_NAME[raw_id] = mapped_name
    print(f"原类别 {raw_id}:{raw_name!r} -> {mapped_name!r}")

unknown_classes = [
    (raw_id, RAW_CLASS_NAMES[raw_id])
    for raw_id, mapped in RAW_ID_TO_TARGET_NAME.items()
    if mapped is None
]
if unknown_classes:
    print("\n[警告] 以下类别没有自动映射，将在预处理中忽略：")
    for item in unknown_classes:
        print(item)


def find_split_directories(dataset_root: Path, split_aliases: list[str]) -> tuple[Path, Path] | None:
    """兼容 Roboflow 常见的两种目录布局。"""
    for split_name in split_aliases:
        candidates = [
            (
                dataset_root / split_name / "images",
                dataset_root / split_name / "labels",
            ),
            (
                dataset_root / "images" / split_name,
                dataset_root / "labels" / split_name,
            ),
        ]

        for images_dir, labels_dir in candidates:
            if images_dir.exists():
                return images_dir, labels_dir

    return None


def list_images(images_dir: Path) -> list[Path]:
    return sorted(
        path
        for path in images_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def label_path_for_image(image_path: Path, images_dir: Path, labels_dir: Path) -> Path:
    relative_path = image_path.relative_to(images_dir)
    return (labels_dir / relative_path).with_suffix(".txt")


def polygon_to_bbox(values: list[float]) -> tuple[float, float, float, float]:
    """
    将 YOLO polygon：
    x1 y1 x2 y2 ... xn yn
    转换为 YOLO bbox：
    x_center y_center width height
    """
    if len(values) < 6 or len(values) % 2 != 0:
        raise ValueError("多边形坐标必须至少包含3个点，且坐标数量必须为偶数")

    xs = values[0::2]
    ys = values[1::2]

    x_min = max(0.0, min(xs))
    y_min = max(0.0, min(ys))
    x_max = min(1.0, max(xs))
    y_max = min(1.0, max(ys))

    width = x_max - x_min
    height = y_max - y_min
    x_center = (x_min + x_max) / 2.0
    y_center = (y_min + y_max) / 2.0

    return x_center, y_center, width, height


def parse_and_convert_label_line(
    line: str,
    raw_label_path: Path,
    line_number: int,
) -> tuple[str | None, dict[str, Any]]:
    """
    支持：
    - YOLO bbox：class x_center y_center width height
    - YOLO polygon：class x1 y1 x2 y2 ... xn yn
    """
    metadata: dict[str, Any] = {
        "source": str(raw_label_path),
        "line_number": line_number,
        "original": line,
        "status": "unknown",
    }

    parts = line.strip().split()
    if not parts:
        metadata["status"] = "empty_line"
        return None, metadata

    try:
        raw_class_id = int(float(parts[0]))
        coordinate_values = [float(value) for value in parts[1:]]
    except ValueError:
        metadata["status"] = "parse_error"
        return None, metadata

    if raw_class_id not in RAW_CLASS_NAMES:
        metadata["status"] = "unknown_class_id"
        return None, metadata

    mapped_name = RAW_ID_TO_TARGET_NAME.get(raw_class_id)

    if mapped_name == "no_leak":
        metadata["status"] = "removed_no_leak"
        return None, metadata

    if mapped_name not in TARGET_CLASS_TO_ID:
        metadata["status"] = "ignored_unmapped_class"
        return None, metadata

    try:
        if len(coordinate_values) == 4:
            x_center, y_center, width, height = coordinate_values
            source_format = "bbox"
        elif len(coordinate_values) >= 6 and len(coordinate_values) % 2 == 0:
            x_center, y_center, width, height = polygon_to_bbox(coordinate_values)
            source_format = "polygon"
        else:
            metadata["status"] = "invalid_coordinate_count"
            return None, metadata
    except Exception as exc:
        metadata["status"] = "conversion_error"
        metadata["error"] = str(exc)
        return None, metadata

    if not (
        0.0 <= x_center <= 1.0
        and 0.0 <= y_center <= 1.0
        and 0.0 < width <= 1.0
        and 0.0 < height <= 1.0
    ):
        metadata["status"] = "invalid_bbox"
        metadata["bbox"] = [x_center, y_center, width, height]
        return None, metadata

    target_class_id = TARGET_CLASS_TO_ID[mapped_name]
    converted_line = (
        f"{target_class_id} "
        f"{x_center:.8f} {y_center:.8f} "
        f"{width:.8f} {height:.8f}"
    )

    metadata["status"] = "converted"
    metadata["source_format"] = source_format
    metadata["target_class"] = mapped_name
    metadata["bbox"] = [x_center, y_center, width, height]

    return converted_line, metadata


def convert_label_file(raw_label_path: Path) -> tuple[list[str], list[dict[str, Any]]]:
    if not raw_label_path.exists():
        return [], [{
            "source": str(raw_label_path),
            "status": "missing_label_file",
        }]

    converted_lines: list[str] = []
    events: list[dict[str, Any]] = []

    for line_number, line in enumerate(
        raw_label_path.read_text(encoding="utf-8", errors="ignore").splitlines(),
        start=1,
    ):
        converted_line, metadata = parse_and_convert_label_line(
            line=line,
            raw_label_path=raw_label_path,
            line_number=line_number,
        )
        events.append(metadata)

        if converted_line is not None:
            converted_lines.append(converted_line)

    return converted_lines, events


def image_class_signature(label_lines: list[str]) -> str:
    """
    用于将 valid 拆成 valid/test 时尽量保持多类别组合分布。
    示例：crack+leak、water、negative。
    """
    class_ids = sorted({
        int(line.split()[0])
        for line in label_lines
        if line.strip()
    })

    if not class_ids:
        return "negative"

    return "+".join(TARGET_CLASSES[class_id] for class_id in class_ids)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


print("预处理函数定义完成。")

## Cell 5：执行自动预处理

In [ ]:
# 清空旧的处理后数据，避免旧文件干扰
if PROCESSED_DATASET_ROOT.exists():
    shutil.rmtree(PROCESSED_DATASET_ROOT)

for split_name in ["train", "val", "test"]:
    (PROCESSED_DATASET_ROOT / split_name / "images").mkdir(
        parents=True,
        exist_ok=True,
    )
    (PROCESSED_DATASET_ROOT / split_name / "labels").mkdir(
        parents=True,
        exist_ok=True,
    )


raw_split_dirs = {
    "train": find_split_directories(
        RAW_DATASET_ROOT,
        ["train"],
    ),
    "valid": find_split_directories(
        RAW_DATASET_ROOT,
        ["valid", "val"],
    ),
    "test": find_split_directories(
        RAW_DATASET_ROOT,
        ["test"],
    ),
}

print("检测到的原始划分目录：")
for split_name, directories in raw_split_dirs.items():
    print(split_name, directories)

if raw_split_dirs["train"] is None:
    raise FileNotFoundError("没有找到 train/images 与 train/labels")

if raw_split_dirs["valid"] is None:
    raise FileNotFoundError("没有找到 valid/images 与 valid/labels")


conversion_events: list[dict[str, Any]] = []
prepared_items_by_raw_split: dict[str, list[dict[str, Any]]] = {
    "train": [],
    "valid": [],
    "test": [],
}


for raw_split_name, directories in raw_split_dirs.items():
    if directories is None:
        continue

    images_dir, labels_dir = directories

    for image_path in list_images(images_dir):
        raw_label_path = label_path_for_image(
            image_path=image_path,
            images_dir=images_dir,
            labels_dir=labels_dir,
        )

        converted_lines, events = convert_label_file(raw_label_path)
        conversion_events.extend(events)

        try:
            with Image.open(image_path) as image:
                image.verify()
        except Exception as exc:
            conversion_events.append({
                "source": str(image_path),
                "status": "unreadable_image",
                "error": str(exc),
            })
            continue

        prepared_items_by_raw_split[raw_split_name].append({
            "image_path": image_path,
            "label_lines": converted_lines,
            "signature": image_class_signature(converted_lines),
            "sha256": sha256_file(image_path),
        })


def copy_items_to_processed(
    items: list[dict[str, Any]],
    target_split: str,
) -> None:
    images_output_dir = PROCESSED_DATASET_ROOT / target_split / "images"
    labels_output_dir = PROCESSED_DATASET_ROOT / target_split / "labels"

    for item in items:
        image_path: Path = item["image_path"]

        # 使用哈希前8位防止不同子目录出现同名文件
        output_stem = f"{image_path.stem}__{item['sha256'][:8]}"
        output_image_path = images_output_dir / f"{output_stem}{image_path.suffix.lower()}"
        output_label_path = labels_output_dir / f"{output_stem}.txt"

        shutil.copy2(image_path, output_image_path)
        output_label_path.write_text(
            "\n".join(item["label_lines"]),
            encoding="utf-8",
        )


# 训练集直接复制
copy_items_to_processed(
    prepared_items_by_raw_split["train"],
    "train",
)

# 如果原数据集已有 test，保留原 valid/test
if prepared_items_by_raw_split["test"]:
    copy_items_to_processed(
        prepared_items_by_raw_split["valid"],
        "val",
    )
    copy_items_to_processed(
        prepared_items_by_raw_split["test"],
        "test",
    )
else:
    # 否则将原 valid 拆成新的 val/test
    valid_items = prepared_items_by_raw_split["valid"]

    if len(valid_items) < 2:
        raise RuntimeError("原 valid 图片不足，无法拆分 valid/test")

    indices = np.arange(len(valid_items))
    signatures = [item["signature"] for item in valid_items]

    signature_counts = Counter(signatures)
    can_stratify = all(count >= 2 for count in signature_counts.values())

    try:
        val_indices, test_indices = train_test_split(
            indices,
            test_size=TEST_RATIO_FROM_VALID,
            random_state=SEED,
            shuffle=True,
            stratify=signatures if can_stratify else None,
        )
    except ValueError as exc:
        print("分层拆分失败，退回固定随机拆分：", exc)
        val_indices, test_indices = train_test_split(
            indices,
            test_size=TEST_RATIO_FROM_VALID,
            random_state=SEED,
            shuffle=True,
            stratify=None,
        )

    val_items = [valid_items[index] for index in val_indices]
    test_items = [valid_items[index] for index in test_indices]

    copy_items_to_processed(val_items, "val")
    copy_items_to_processed(test_items, "test")


processed_data_yaml = {
    "path": str(PROCESSED_DATASET_ROOT.resolve()).replace("\\", "/"),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {
        0: "crack",
        1: "water",
        2: "leak",
    },
}

PROCESSED_DATA_YAML_PATH = PROCESSED_DATASET_ROOT / "data.yaml"
with PROCESSED_DATA_YAML_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        processed_data_yaml,
        file,
        allow_unicode=True,
        sort_keys=False,
    )

events_df = pd.DataFrame(conversion_events)
events_csv_path = REPORT_ROOT / "preprocess_events.csv"
events_df.to_csv(events_csv_path, index=False, encoding="utf-8-sig")

print("处理完成。")
print("处理后 data.yaml：", PROCESSED_DATA_YAML_PATH)
print("转换事件报告：", events_csv_path)

if not events_df.empty:
    display(events_df["status"].value_counts().rename_axis("status").to_frame("count"))

## Cell 6：审计处理后的数据集

In [ ]:
def audit_processed_dataset(dataset_root: Path) -> dict[str, Any]:
    report: dict[str, Any] = {
        "splits": {},
        "invalid_labels": [],
        "cross_split_duplicates": [],
    }

    hash_occurrences: dict[str, list[dict[str, str]]] = defaultdict(list)

    for split_name in ["train", "val", "test"]:
        images_dir = dataset_root / split_name / "images"
        labels_dir = dataset_root / split_name / "labels"

        image_paths = list_images(images_dir)
        class_counts = Counter()
        image_target_counts = Counter()
        empty_label_images = 0

        for image_path in image_paths:
            image_hash = sha256_file(image_path)
            hash_occurrences[image_hash].append({
                "split": split_name,
                "image": image_path.name,
            })

            label_path = labels_dir / f"{image_path.stem}.txt"

            if not label_path.exists():
                report["invalid_labels"].append({
                    "image": str(image_path),
                    "label": str(label_path),
                    "error": "missing_label_file",
                })
                continue

            label_lines = [
                line.strip()
                for line in label_path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                ).splitlines()
                if line.strip()
            ]

            if not label_lines:
                empty_label_images += 1
                image_target_counts[0] += 1
                continue

            valid_targets_in_image = 0

            for line_number, line in enumerate(label_lines, start=1):
                parts = line.split()

                try:
                    if len(parts) != 5:
                        raise ValueError("处理后的检测标签必须正好5列")

                    class_id = int(float(parts[0]))
                    x_center, y_center, width, height = map(float, parts[1:])

                    if class_id not in {0, 1, 2}:
                        raise ValueError(f"类别ID越界：{class_id}")

                    if not (
                        0.0 <= x_center <= 1.0
                        and 0.0 <= y_center <= 1.0
                        and 0.0 < width <= 1.0
                        and 0.0 < height <= 1.0
                    ):
                        raise ValueError("BBox坐标越界")

                    class_counts[TARGET_CLASSES[class_id]] += 1
                    valid_targets_in_image += 1

                except Exception as exc:
                    report["invalid_labels"].append({
                        "label": str(label_path),
                        "line_number": line_number,
                        "content": line,
                        "error": str(exc),
                    })

            image_target_counts[valid_targets_in_image] += 1

        report["splits"][split_name] = {
            "images": len(image_paths),
            "empty_negative_images": empty_label_images,
            "class_box_counts": dict(class_counts),
            "targets_per_image": {
                str(key): value
                for key, value in sorted(image_target_counts.items())
            },
        }

    for image_hash, occurrences in hash_occurrences.items():
        split_names = {item["split"] for item in occurrences}
        if len(split_names) > 1:
            report["cross_split_duplicates"].append({
                "sha256": image_hash,
                "occurrences": occurrences,
            })

    return report


audit_report = audit_processed_dataset(PROCESSED_DATASET_ROOT)

audit_report_path = REPORT_ROOT / "dataset_audit.json"
audit_report_path.write_text(
    json.dumps(audit_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(audit_report["splits"], ensure_ascii=False, indent=2))
print("\n无效标签数量：", len(audit_report["invalid_labels"]))
print("跨划分重复图片数量：", len(audit_report["cross_split_duplicates"]))
print("完整审计报告：", audit_report_path)

if audit_report["invalid_labels"]:
    raise RuntimeError("数据集中仍存在无效标签，请先检查 dataset_audit.json")

if audit_report["cross_split_duplicates"]:
    raise RuntimeError(
        "train/val/test 存在完全重复图片。"
        "为避免指标虚高，请先删除跨划分重复项。"
    )

print("\n数据审计通过。")

## Cell 7：可视化随机标注样本

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import ImageDraw, ImageFont


DISPLAY_CLASS_NAMES = {
    0: "crack",
    1: "water",
    2: "leak",
}


def draw_yolo_labels(
    image_path: Path,
    label_path: Path,
) -> Image.Image:
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    width, height = image.size

    label_lines = (
        label_path.read_text(encoding="utf-8").splitlines()
        if label_path.exists()
        else []
    )

    for line in label_lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        class_id = int(parts[0])
        x_center, y_center, box_width, box_height = map(float, parts[1:])

        x_min = (x_center - box_width / 2) * width
        y_min = (y_center - box_height / 2) * height
        x_max = (x_center + box_width / 2) * width
        y_max = (y_center + box_height / 2) * height

        draw.rectangle(
            [x_min, y_min, x_max, y_max],
            width=max(2, int(min(width, height) * 0.004)),
        )
        draw.text(
            (x_min, max(0, y_min - 16)),
            DISPLAY_CLASS_NAMES[class_id],
        )

    return image


train_images_dir = PROCESSED_DATASET_ROOT / "train" / "images"
train_labels_dir = PROCESSED_DATASET_ROOT / "train" / "labels"

sample_image_paths = random.Random(SEED).sample(
    list_images(train_images_dir),
    k=min(6, len(list_images(train_images_dir))),
)

fig, axes = plt.subplots(
    nrows=len(sample_image_paths),
    ncols=1,
    figsize=(12, 6 * len(sample_image_paths)),
)

if len(sample_image_paths) == 1:
    axes = [axes]

for axis, image_path in zip(axes, sample_image_paths):
    label_path = train_labels_dir / f"{image_path.stem}.txt"
    annotated_image = draw_yolo_labels(image_path, label_path)
    axis.imshow(annotated_image)
    axis.set_title(image_path.name)
    axis.axis("off")

plt.tight_layout()
plt.show()

## Cell 8：封装 LoRA 训练函数

In [ ]:
from ultralytics import YOLO


def train_yolo11_lora(
    epochs: int,
    run_name: str,
):
    model = YOLO(BASE_MODEL)

    print("=" * 80)
    print("开始 YOLO11s + LoRA 训练")
    print("epochs:", epochs)
    print("run_name:", run_name)
    print("data:", PROCESSED_DATA_YAML_PATH)
    print("=" * 80)

    results = model.train(
        data=str(PROCESSED_DATA_YAML_PATH),
        epochs=epochs,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        optimizer=OPTIMIZER,
        lr0=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        patience=PATIENCE,
        amp=True,
        seed=SEED,
        deterministic=True,
        cache=False,
        plots=True,
        project=str(RUNS_ROOT),
        name=run_name,
        exist_ok=True,

        # LoRA
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        lora_gradient_checkpointing=LORA_GRADIENT_CHECKPOINTING,
    )

    return model, results


print("LoRA训练函数已定义。")

## Cell 9：3 epochs 冒烟训练

本轮只验证以下内容：

- 数据集是否能正常读取；
- 多边形转 BBox 是否成功；
- `no leak` 是否已移除；
- CUDA 是否正常；
- YOLO-Master 是否真正识别 LoRA 参数；
- 权重能否正常保存。

该权重不能作为客户最终演示模型。

In [ ]:
smoke_model = None
smoke_results = None

if RUN_SMOKE_TEST:
    smoke_model, smoke_results = train_yolo11_lora(
        epochs=SMOKE_EPOCHS,
        run_name="yolo11s_lora_smoke",
    )

    SMOKE_BEST_MODEL_PATH = (
        RUNS_ROOT
        / "yolo11s_lora_smoke"
        / "weights"
        / "best.pt"
    )

    print("冒烟训练 best.pt：", SMOKE_BEST_MODEL_PATH)

    if not SMOKE_BEST_MODEL_PATH.exists():
        raise FileNotFoundError("冒烟训练完成后没有找到 best.pt")
else:
    print("RUN_SMOKE_TEST=False，跳过冒烟训练。")

## Cell 10：150 epochs 正式 LoRA 训练

确认冒烟训练成功后，将 Cell 1 中：

```python
RUN_FULL_TRAINING = False
```

改成：

```python
RUN_FULL_TRAINING = True
```

然后重新运行本 Cell。

> 正式训练可能耗时较长。训练过程中将根据验证集结果保存 `best.pt`。

In [ ]:
full_model = None
full_results = None

if RUN_FULL_TRAINING:
    full_model, full_results = train_yolo11_lora(
        epochs=FULL_EPOCHS,
        run_name="yolo11s_lora_full",
    )

    FINAL_BEST_MODEL_PATH = (
        RUNS_ROOT
        / "yolo11s_lora_full"
        / "weights"
        / "best.pt"
    )

    print("正式训练 best.pt：", FINAL_BEST_MODEL_PATH)

    if not FINAL_BEST_MODEL_PATH.exists():
        raise FileNotFoundError("正式训练完成后没有找到 best.pt")
else:
    FINAL_BEST_MODEL_PATH = (
        RUNS_ROOT
        / "yolo11s_lora_full"
        / "weights"
        / "best.pt"
    )
    print(
        "RUN_FULL_TRAINING=False，未执行正式训练。\n"
        "冒烟验证通过后，请将 RUN_FULL_TRAINING 改为 True。"
    )

## Cell 11：选择待评估模型

In [ ]:
# 优先评估正式模型；若正式模型尚不存在，则评估冒烟模型
if FINAL_BEST_MODEL_PATH.exists():
    MODEL_TO_EVALUATE = FINAL_BEST_MODEL_PATH
elif (
    RUNS_ROOT / "yolo11s_lora_smoke" / "weights" / "best.pt"
).exists():
    MODEL_TO_EVALUATE = (
        RUNS_ROOT
        / "yolo11s_lora_smoke"
        / "weights"
        / "best.pt"
    )
    print("[提示] 当前使用的是3 epochs冒烟模型，指标不代表最终效果。")
else:
    raise FileNotFoundError("没有找到可评估的 best.pt")

print("待评估模型：", MODEL_TO_EVALUATE)

## Cell 12：在独立 test 集上评估

In [ ]:
evaluation_model = YOLO(str(MODEL_TO_EVALUATE))

metrics = evaluation_model.val(
    data=str(PROCESSED_DATA_YAML_PATH),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=max(1, min(BATCH_SIZE, 8)),
    device=DEVICE,
    workers=WORKERS,
    conf=0.001,
    iou=0.6,
    plots=True,
    project=str(RUNS_ROOT),
    name="yolo11s_lora_test_evaluation",
    exist_ok=True,
)

summary_metrics = {
    "model_path": str(MODEL_TO_EVALUATE.resolve()),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "preprocess_ms": float(metrics.speed.get("preprocess", 0.0)),
    "inference_ms": float(metrics.speed.get("inference", 0.0)),
    "postprocess_ms": float(metrics.speed.get("postprocess", 0.0)),
}

metrics_output_path = REPORT_ROOT / "lora_test_metrics.json"
metrics_output_path.write_text(
    json.dumps(summary_metrics, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary_metrics, ensure_ascii=False, indent=2))
print("\n指标已保存：", metrics_output_path)

### 指标解释

- `Precision`：模型检出的目标中，有多少是真实目标；
- `Recall`：所有真实目标中，模型成功找到多少；
- `mAP50`：在 IoU=0.50 条件下的综合检测能力；
- `mAP50-95`：在更严格的多个 IoU 阈值下的综合检测能力；
- 图片框后的百分比是当前检测框的模型置信度，不是模型整体准确率。

## Cell 13：查看各类别指标

In [ ]:
class_rows = []

for class_id, class_map50_95 in enumerate(metrics.box.maps):
    class_rows.append({
        "class_id": class_id,
        "class_name": TARGET_CLASSES[class_id],
        "mAP50_95": float(class_map50_95),
    })

class_metrics_df = pd.DataFrame(class_rows)
display(class_metrics_df)

class_metrics_path = REPORT_ROOT / "lora_per_class_metrics.csv"
class_metrics_df.to_csv(
    class_metrics_path,
    index=False,
    encoding="utf-8-sig",
)

print("类别指标已保存：", class_metrics_path)

## Cell 14：对测试图片进行预测并保存结果

In [ ]:
test_images_dir = PROCESSED_DATASET_ROOT / "test" / "images"
test_image_paths = list_images(test_images_dir)

if not test_image_paths:
    raise RuntimeError("test/images 中没有图片")

demo_sample_paths = random.Random(SEED).sample(
    test_image_paths,
    k=min(12, len(test_image_paths)),
)

prediction_results = evaluation_model.predict(
    source=[str(path) for path in demo_sample_paths],
    imgsz=IMAGE_SIZE,
    conf=0.25,
    iou=0.50,
    device=DEVICE,
    save=True,
    project=str(DEMO_ROOT),
    name="test_predictions",
    exist_ok=True,
    verbose=False,
)

print("预测结果保存目录：", DEMO_ROOT / "test_predictions")

## Cell 15：在 Notebook 中显示预测结果

In [ ]:
prediction_output_dir = DEMO_ROOT / "test_predictions"

predicted_image_paths = [
    path
    for path in list_images(prediction_output_dir)
]

if not predicted_image_paths:
    print("没有找到已保存的预测图片。")
else:
    selected_paths = predicted_image_paths[: min(8, len(predicted_image_paths))]

    fig, axes = plt.subplots(
        nrows=len(selected_paths),
        ncols=1,
        figsize=(14, 7 * len(selected_paths)),
    )

    if len(selected_paths) == 1:
        axes = [axes]

    for axis, image_path in zip(axes, selected_paths):
        image = Image.open(image_path).convert("RGB")
        axis.imshow(image)
        axis.set_title(image_path.name)
        axis.axis("off")

    plt.tight_layout()
    plt.show()

## Cell 16：输出结构化检测结果

In [ ]:
structured_predictions: list[dict[str, Any]] = []

for source_path, result in zip(demo_sample_paths, prediction_results):
    image_record = {
        "image": str(source_path),
        "detections": [],
    }

    if result.boxes is not None:
        for box in result.boxes:
            class_id = int(box.cls.item())
            confidence = float(box.conf.item())
            bbox_xyxy = [
                round(float(value), 2)
                for value in box.xyxy[0].tolist()
            ]

            image_record["detections"].append({
                "class_id": class_id,
                "class_name": result.names[class_id],
                "confidence": round(confidence, 4),
                "bbox_xyxy": bbox_xyxy,
            })

    if image_record["detections"]:
        detected_classes = {
            item["class_name"]
            for item in image_record["detections"]
        }

        if "leak" in detected_classes:
            image_record["front_end_status"] = "检测到疑似泄漏"
        else:
            image_record["front_end_status"] = (
                "未检测到明确泄漏，但发现相关异常"
            )
    else:
        image_record["front_end_status"] = (
            "当前图像中未检测到超过阈值的明显异常"
        )

    structured_predictions.append(image_record)

structured_output_path = REPORT_ROOT / "demo_predictions.json"
structured_output_path.write_text(
    json.dumps(
        structured_predictions,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(json.dumps(structured_predictions[:3], ensure_ascii=False, indent=2))
print("\n结构化预测结果：", structured_output_path)

## Cell 17：可选——导出 ONNX

In [ ]:
EXPORT_ONNX = False

if EXPORT_ONNX:
    export_model = YOLO(str(MODEL_TO_EVALUATE))
    exported_path = export_model.export(
        format="onnx",
        imgsz=IMAGE_SIZE,
        simplify=True,
        dynamic=False,
        half=False,
        device=DEVICE,
    )
    print("ONNX导出结果：", exported_path)
else:
    print("EXPORT_ONNX=False，跳过ONNX导出。")

## 最终输出位置

处理后数据集：

```text
pipeline_leakage1_lora_workspace/processed_dataset/
```

3 epochs 冒烟模型：

```text
pipeline_leakage1_lora_workspace/runs/yolo11s_lora_smoke/weights/best.pt
```

150 epochs 正式 LoRA 模型：

```text
pipeline_leakage1_lora_workspace/runs/yolo11s_lora_full/weights/best.pt
```

测试集指标：

```text
pipeline_leakage1_lora_workspace/reports/lora_test_metrics.json
pipeline_leakage1_lora_workspace/reports/lora_per_class_metrics.csv
```

带框图片：

```text
pipeline_leakage1_lora_workspace/demo_predictions/test_predictions/
```

## 前端最终输出逻辑

- 检测到 `leak`：显示“检测到疑似泄漏”；
- 未检测到 `leak`，但检测到 `crack` 或 `water`：显示“未检测到明确泄漏，但发现相关异常”；
- 三类均未检测到：显示“当前图像中未检测到明显异常”。

`no leak` 不作为检测框类别。